# Telco Customer Churn EDA

This notebook is a lightweight exploratory companion to the training pipeline. All production feature engineering is implemented in `scripts/preprocessing.py` so it is reproduced at inference time.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(DATA_PATH)
df.shape, df.dtypes

In [ ]:
df.head()

## Mistyped Columns

`TotalCharges` is mistyped as `object` because several rows contain blank strings. It should be converted with `pd.to_numeric(..., errors='coerce')` and imputed inside the Pipeline.

In [ ]:
df['Churn'].value_counts(normalize=True).rename('share')

In [ ]:
df['Churn'].value_counts().plot(kind='bar', title='Churn Distribution')
plt.ylabel('Customers')
plt.tight_layout()

In [ ]:
eda_df = df.copy()
eda_df['TotalCharges'] = pd.to_numeric(eda_df['TotalCharges'], errors='coerce')
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
eda_df[numeric_cols].hist(figsize=(10, 7), bins=30)
plt.tight_layout()

In [ ]:
missing_report = pd.DataFrame({
    'explicit_na': df.isna().sum(),
    'blank_strings': df.astype(str).apply(lambda s: s.str.strip().eq('').sum()),
})
missing_report[missing_report.sum(axis=1) > 0]

In [ ]:
for col in ['Contract', 'PaymentMethod', 'InternetService']:
    display(pd.crosstab(df[col], df['Churn'], normalize='index').round(3))

## Initial Insights

- Month-to-month customers churn much more often than one-year or two-year contract customers.
- Electronic check users show elevated churn compared with mailed check or automatic payment methods.
- Fiber optic internet customers have higher churn than DSL customers, suggesting plan cost, competition, or service quality may matter.